In [5]:
import time
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score
import great_expectations as ge
from sklearn.preprocessing import StandardScaler

In [6]:
def process_stream_in_batches(data_path, chunksize=100000):
    print(f"\n==================================================")
    print(f"STREAMING PIPELINE (CHUNK SIZE: {chunksize:,}) ON: {data_path}")
    print(f"==================================================")
    
    t_start_total = time.time()
    
    # Accumulated prediction & target lists across streams
    all_y_true = []
    all_y_pred = []
    
    total_repair_time = 0.0
    total_validation_time = 0.0
    batch_count = 0
    
    # 1. Stream file in micro-batches using pandas chunking
    for chunk in pd.read_csv(data_path, chunksize=chunksize):
        batch_count += 1
        
        # Filter relevant transaction types
        df_filtered = chunk[chunk['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
        if df_filtered.empty:
            continue
            
        # ----------------------------------------------------
        # STAGE 1: DECLARATIVE VALIDATION GATEWAY
        # ----------------------------------------------------
        t_v_start = time.time()
        
        # Fast vector validation for streaming micro-batches
        target_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
        has_nulls = df_filtered[target_cols].isna().any().any()
        has_negatives = (df_filtered['amount'] < 0).any()
        
        validation_passed = not (has_nulls or has_negatives)
        t_v_end = time.time()
        total_validation_time += (t_v_end - t_v_start)
        
        # ----------------------------------------------------
        # STAGE 2: PROBABILISTIC REPAIR LAYER (HoloClean Logic)
        # ----------------------------------------------------
        t_r_start = time.time()
        df_repaired = df_filtered.copy()
        
        if not validation_passed:
            # Relational Denial Constraints
            mask_old = df_repaired['oldbalanceOrg'].isna() & df_repaired['newbalanceOrig'].notna() & df_repaired['amount'].notna()
            df_repaired.loc[mask_old, 'oldbalanceOrg'] = df_repaired.loc[mask_old, 'newbalanceOrig'] + df_repaired.loc[mask_old, 'amount']
            
            mask_new = df_repaired['newbalanceOrig'].isna() & df_repaired['oldbalanceOrg'].notna() & df_repaired['amount'].notna()
            df_repaired.loc[mask_new, 'newbalanceOrig'] = df_repaired.loc[mask_new, 'oldbalanceOrg'] - df_repaired.loc[mask_new, 'amount']
            
            mask_amt = df_repaired['amount'].isna() & df_repaired['oldbalanceOrg'].notna() & df_repaired['newbalanceOrig'].notna()
            df_repaired.loc[mask_amt, 'amount'] = df_repaired.loc[mask_amt, 'oldbalanceOrg'] - df_repaired.loc[mask_amt, 'newbalanceOrig']
            
            # Fallback imputations
            for col in target_cols:
                if df_repaired[col].isna().any():
                    df_repaired[col] = df_repaired[col].fillna(df_repaired[col].median())
                    
        t_r_end = time.time()
        total_repair_time += (t_r_end - t_r_start)
        
        # ----------------------------------------------------
        # STAGE 3: DOWNSTREAM MODELING (Engineered & Scaled)
        # ----------------------------------------------------
        # Domain Feature Engineering
        df_repaired['balance_diff_orig'] = (
            df_repaired['oldbalanceOrg'] - df_repaired['amount'] - df_repaired['newbalanceOrig']
        )
        # Fraud indicator: Origin account completely emptied
        df_repaired['orig_zero_after'] = ((df_repaired['oldbalanceOrg'] > 0) & (df_repaired['newbalanceOrig'] == 0)).astype(int)
        
        # Fraud indicator: Destination account balance unchanged despite transfer
        df_repaired['dest_no_change'] = ((df_repaired['amount'] > 0) & (df_repaired['oldbalanceDest'] == df_repaired['newbalanceDest'])).astype(int)

        feature_cols = [
            'amount', 'oldbalanceOrg', 'newbalanceOrig', 
            'oldbalanceDest', 'newbalanceDest', 'balance_diff_orig',
            'orig_zero_after', 'dest_no_change'
        ]

        # Standardize features to fix scale disparities across tree splits
        from sklearn.preprocessing import StandardScaler
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(df_repaired[feature_cols].fillna(0))
        y = df_repaired['isFraud']

        # Fit model on normalized feature space
        model = IsolationForest(
            n_estimators=100,
            contamination=max(y.mean(), 0.001),
            random_state=42,
            n_jobs=-1
        )

        preds = model.fit_predict(X_scaled)
        preds_binary = (preds == -1).astype(int)

        all_y_true.extend(y.tolist())
        all_y_pred.extend(preds_binary.tolist())
    t_end_total = time.time()
    
    # Calculate Overall Stream Performance Metrics
    overall_precision = precision_score(all_y_true, all_y_pred, pos_label=1, zero_division=0)
    overall_recall = recall_score(all_y_true, all_y_pred, pos_label=1, zero_division=0)
    overall_f1 = f1_score(all_y_true, all_y_pred, pos_label=1, zero_division=0)
    total_latency = t_end_total - t_start_total
    
    print(f"\n\n=== STREAMING EVALUATION RESULTS ===")
    print(f"Total Batches Processed: {batch_count}")
    print(f"Total Stream Time (Delta T): {total_latency:.4f} seconds")
    print(f"Validation Overhead: {total_validation_time:.4f} seconds")
    print(f"Repair Overhead (Delta T_r): {total_repair_time:.4f} seconds")
    print(f"Overall Stream Precision: {overall_precision:.4f}")
    print(f"Overall Stream Recall: {overall_recall:.4f}")
    print(f"Overall Stream F1-Score: {overall_f1:.4f}")

if __name__ == "__main__":
    process_stream_in_batches('Dataset.csv', chunksize=100000)


STREAMING PIPELINE (CHUNK SIZE: 100,000) ON: Dataset.csv


=== STREAMING EVALUATION RESULTS ===
Total Batches Processed: 64
Total Stream Time (Delta T): 100.1044 seconds
Validation Overhead: 0.2456 seconds
Repair Overhead (Delta T_r): 0.5197 seconds
Overall Stream Precision: 0.1968
Overall Stream Recall: 0.2085
Overall Stream F1-Score: 0.2025


In [7]:
process_stream_in_batches('paysim_noise_5.csv', chunksize=100000)
process_stream_in_batches('paysim_noise_10.csv', chunksize=100000)
process_stream_in_batches('paysim_noise_20.csv', chunksize=100000)


STREAMING PIPELINE (CHUNK SIZE: 100,000) ON: paysim_noise_5.csv


=== STREAMING EVALUATION RESULTS ===
Total Batches Processed: 28
Total Stream Time (Delta T): 69.9336 seconds
Validation Overhead: 0.1187 seconds
Repair Overhead (Delta T_r): 1.5183 seconds
Overall Stream Precision: 0.2150
Overall Stream Recall: 0.2204
Overall Stream F1-Score: 0.2177

STREAMING PIPELINE (CHUNK SIZE: 100,000) ON: paysim_noise_10.csv


=== STREAMING EVALUATION RESULTS ===
Total Batches Processed: 28
Total Stream Time (Delta T): 67.1979 seconds
Validation Overhead: 0.1077 seconds
Repair Overhead (Delta T_r): 1.5720 seconds
Overall Stream Precision: 0.1749
Overall Stream Recall: 0.1791
Overall Stream F1-Score: 0.1770

STREAMING PIPELINE (CHUNK SIZE: 100,000) ON: paysim_noise_20.csv


=== STREAMING EVALUATION RESULTS ===
Total Batches Processed: 28
Total Stream Time (Delta T): 66.5979 seconds
Validation Overhead: 0.1102 seconds
Repair Overhead (Delta T_r): 1.5060 seconds
Overall Stream Precision: 0.1492
Over